# NB02 — Data Transformation

## Purpose

This notebook prepares the original World Bank API responses collected in NB01 for analysis.

It will:

1. load the two JSON files saved under `data/raw/`;
2. inspect their metadata and record structure;
3. extract the fields needed for analysis;
4. combine the two indicators into one tidy table;
5. validate countries, years, missing values, and duplicate keys;
6. save the prepared table under `data/processed/`.

The transformation first creates a long intermediate table with one row
per country, year, and indicator. It then reshapes those records into the
final analysis-ready table, where one row represents one country in one
year and the two indicators appear in separate columns.

Data collection is not repeated in this notebook.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

RAW_FILES = {
    "inflation": RAW_DATA_DIR / "world_bank_inflation.json",
    "youth_unemployment": (
        RAW_DATA_DIR / "world_bank_youth_unemployment.json"
    ),
}

OUTPUT_FILE = (
    PROCESSED_DATA_DIR / "country_year_indicators.csv"
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
print("Raw data directory:", RAW_DATA_DIR.resolve())
print("Processed data directory:", PROCESSED_DATA_DIR.resolve())

for source_name, path in RAW_FILES.items():
    print(
        f"{source_name}: "
        f"{path.name} — exists: {path.exists()}"
    )

print("Output file:", OUTPUT_FILE.name)

Raw data directory: /files/assignments/final-project/data/raw
Processed data directory: /files/assignments/final-project/data/processed
inflation: world_bank_inflation.json — exists: True
youth_unemployment: world_bank_youth_unemployment.json — exists: True
Output file: country_year_indicators.csv


## Loading the preserved raw responses

NB02 reads the two JSON files saved by NB01 rather than sending new API
requests. This keeps data collection and transformation separate and
ensures that the preparation is applied to the exact raw responses stored
under `data/raw/`.

In [4]:
raw_payloads = {}

for source_name, path in RAW_FILES.items():
    with path.open("r", encoding="utf-8-sig") as file:
        raw_payloads[source_name] = json.load(file)

print("Loaded sources:", list(raw_payloads))

Loaded sources: ['inflation', 'youth_unemployment']


In [5]:
for source_name, payload in raw_payloads.items():
    print(source_name)
    print("  Outer type:", type(payload))
    print("  Outer elements:", len(payload))
    print("  Metadata type:", type(payload[0]))
    print("  Records type:", type(payload[1]))
    print("  Record count:", len(payload[1]))

inflation
  Outer type: <class 'list'>
  Outer elements: 2
  Metadata type: <class 'dict'>
  Records type: <class 'list'>
  Record count: 48
youth_unemployment
  Outer type: <class 'list'>
  Outer elements: 2
  Metadata type: <class 'dict'>
  Records type: <class 'list'>
  Record count: 48


In [6]:
for source_name, payload in raw_payloads.items():
    metadata, records = payload

    print(source_name)
    print("  Metadata total:", metadata["total"])
    print("  Loaded records:", len(records))
    print("  Last updated:", metadata.get("lastupdated"))

inflation
  Metadata total: 48
  Loaded records: 48
  Last updated: 2026-07-13
youth_unemployment
  Metadata total: 48
  Loaded records: 48
  Last updated: 2026-07-13


In [7]:
for source_name, payload in raw_payloads.items():
    _, records = payload

    print(f"\n{source_name}:")
    print(records[0])


inflation:
{'indicator': {'id': 'FP.CPI.TOTL.ZG', 'value': 'Inflation, consumer prices (annual %)'}, 'country': {'id': 'GB', 'value': 'United Kingdom'}, 'countryiso3code': 'GBR', 'date': '2025', 'value': 3.88306881626, 'unit': '', 'obs_status': '', 'decimal': 1}

youth_unemployment:
{'indicator': {'id': 'SL.UEM.1524.ZS', 'value': 'Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)'}, 'country': {'id': 'GB', 'value': 'United Kingdom'}, 'countryiso3code': 'GBR', 'date': '2025', 'value': 14.647, 'unit': '', 'obs_status': '', 'decimal': 1}


In [8]:
for source_name, payload in raw_payloads.items():
    _, records = payload

    print(
        source_name,
        sorted(records[0].keys()),
    )

inflation ['country', 'countryiso3code', 'date', 'decimal', 'indicator', 'obs_status', 'unit', 'value']
youth_unemployment ['country', 'countryiso3code', 'date', 'decimal', 'indicator', 'obs_status', 'unit', 'value']


## Selecting and restructuring the analytical fields

The raw API records contain several nested and administrative fields. The transformation retains the country code and name, year, indicator identity, and observed value.

A long intermediate table is first created with one row per country, year, and indicator. It is then reshaped into the final analysis-ready table with one row per country-year and separate columns for inflation and youth unemployment.

This final structure allows NB03 to compare the two variables directly without repeating preparation work.

In [9]:
for source_name, payload in raw_payloads.items():
    _, records = payload

    countries = sorted({
        record["country"]["value"]
        for record in records
    })

    indicators = sorted({
        record["indicator"]["value"]
        for record in records
    })

    print(source_name)
    print("  Country labels:", countries)
    print("  Indicator labels:", indicators)

inflation
  Country labels: ['Italy', 'Turkiye', 'United Kingdom']
  Indicator labels: ['Inflation, consumer prices (annual %)']
youth_unemployment
  Country labels: ['Italy', 'Turkiye', 'United Kingdom']
  Indicator labels: ['Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)']


In [10]:
rows = []

for payload in raw_payloads.values():
    _, records = payload

    for record in records:
        rows.append({
            "country_code": record["countryiso3code"],
            "country": record["country"]["value"],
            "year": record["date"],
            "indicator_code": record["indicator"]["id"],
            "indicator": record["indicator"]["value"],
            "value": record["value"],
        })

print("Extracted rows:", len(rows))

Extracted rows: 96


### Inspecting the long intermediate table

The preview displays one observation from each country-indicator
combination. This makes it possible to confirm that both indicators and
all three countries are represented without displaying all 96 rows.

In [11]:
long_df = pd.DataFrame(rows)

long_preview = (
    long_df
    .groupby(
        ["indicator_code", "country_code"],
        sort=False,
    )
    .head(1)
    .reset_index(drop=True)
)

long_preview

,country_code,country,year,indicator_code,indicator,value
0,GBR,United Kingdom,2025,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",3.883069
1,ITA,Italy,2025,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",1.531668
2,TUR,Turkiye,2025,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",34.881163
3,GBR,United Kingdom,2025,SL.UEM.1524.ZS,"Unemployment, youth total (% of total labor fo...",14.647000
4,ITA,Italy,2025,SL.UEM.1524.ZS,"Unemployment, youth total (% of total labor fo...",20.545000
5,TUR,Turkiye,2025,SL.UEM.1524.ZS,"Unemployment, youth total (% of total labor fo...",16.042000


In [12]:
print("Shape:", long_df.shape)
print("\nColumn types:")
print(long_df.dtypes)

print("\nUnique indicators:")
print(long_df["indicator_code"].value_counts())

print("\nRows per country:")
print(long_df["country_code"].value_counts())

Shape: (96, 6)

Column types:
country_code       object
country            object
year               object
indicator_code     object
indicator          object
value             float64
dtype: object

Unique indicators:
indicator_code
FP.CPI.TOTL.ZG    48
SL.UEM.1524.ZS    48
Name: count, dtype: int64

Rows per country:
country_code
GBR    32
ITA    32
TUR    32
Name: count, dtype: int64


### Converting values to numeric types

The API stores the year as text, so it is converted to an integer before
the country-year table is created. The observation values are also
confirmed as numeric. Using `errors="raise"` stops the transformation if
an unexpected non-numeric value is encountered rather than silently
converting or discarding it.

In [13]:
long_df["year"] = pd.to_numeric(
    long_df["year"],
    errors="raise",
).astype("int64")

long_df["value"] = pd.to_numeric(
    long_df["value"],
    errors="raise",
)

long_df.dtypes

country_code       object
country            object
year                int64
indicator_code     object
indicator          object
value             float64
dtype: object

### Validating the long-table keys

Before reshaping, the required fields are checked for missing values and
each country-year-indicator combination is checked for uniqueness. A
duplicate combination would make a simple pivot ambiguous because more
than one value would exist for the same output position.

In [14]:
key_columns = [
    "country_code",
    "year",
    "indicator_code",
]

duplicate_rows = long_df.duplicated(
    subset=key_columns
).sum()

missing_values = long_df[
    key_columns + ["country", "value"]
].isna().sum()

print("Duplicate key rows:", duplicate_rows)
print("\nMissing values:")
print(missing_values)

if duplicate_rows != 0:
    raise ValueError(
        "Duplicate country-year-indicator rows were found."
    )

if missing_values.sum() != 0:
    raise ValueError(
        "Missing values were found in required fields."
    )

Duplicate key rows: 0

Missing values:
country_code      0
year              0
indicator_code    0
country           0
value             0
dtype: int64


### Reshaping the indicators

Because each country-year-indicator combination was confirmed to be unique, `pivot()` is used rather than `pivot_table()`. This reshapes the observations without aggregating or averaging any values.

In [15]:
processed_df = (
    long_df
    .pivot(
        index=["country_code", "country", "year"],
        columns="indicator_code",
        values="value",
    )
    .reset_index()
)

In [16]:

processed_df = processed_df.rename(
    columns={
        "FP.CPI.TOTL.ZG": "inflation_pct",
        "SL.UEM.1524.ZS": "youth_unemployment_pct",
    }
)

processed_df.columns.name = None

processed_df = processed_df.sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

processed_preview = (
    processed_df
    .groupby(
        "country_code",
        sort=False,
    )
    .head(2)
    .reset_index(drop=True)
)

processed_preview

,country_code,country,year,inflation_pct,youth_unemployment_pct
0,GBR,United Kingdom,2010,2.492655,20.328
1,GBR,United Kingdom,2011,3.856112,21.689
2,ITA,Italy,2010,1.525516,27.910
3,ITA,Italy,2011,2.780633,29.145
4,TUR,Turkiye,2010,8.566444,21.578
5,TUR,Turkiye,2011,6.471880,18.208


### Validating the final country-year table

The final table should contain one unique row for each country and year.
With three countries and 16 common years, this implies 48 rows. The
following checks confirm the expected columns, balanced country coverage,
absence of missing values, and absence of duplicate country-year keys.

In [17]:
required_columns = [
    "country_code",
    "country",
    "year",
    "inflation_pct",
    "youth_unemployment_pct",
]

print("Shape:", processed_df.shape)
print("Columns:", list(processed_df.columns))
print(
    "Year range:",
    f"{processed_df['year'].min()}–"
    f"{processed_df['year'].max()}"
)

print("\nRows per country:")
print(processed_df["country_code"].value_counts())

print("\nMissing values:")
print(processed_df[required_columns].isna().sum())

duplicate_country_years = processed_df.duplicated(
    subset=["country_code", "year"]
).sum()

print(
    "\nDuplicate country-year rows:",
    duplicate_country_years,
)

if list(processed_df.columns) != required_columns:
    raise ValueError("The final columns are not as expected.")

if processed_df.shape != (48, 5):
    raise ValueError("The final table does not contain 48 rows.")

if duplicate_country_years != 0:
    raise ValueError("Duplicate country-year rows were found.")

if processed_df[required_columns].isna().sum().sum() != 0:
    raise ValueError("Missing values remain in the final table.")

Shape: (48, 5)
Columns: ['country_code', 'country', 'year', 'inflation_pct', 'youth_unemployment_pct']
Year range: 2010–2025

Rows per country:
country_code
GBR    16
ITA    16
TUR    16
Name: count, dtype: int64

Missing values:
country_code              0
country                   0
year                      0
inflation_pct             0
youth_unemployment_pct    0
dtype: int64

Duplicate country-year rows: 0


### Confirming country coverage

The final table should provide the same annual coverage for every country.
The following summary checks the first year, last year, and number of
country-year observations without beginning the exploratory analysis
reserved for NB03.

In [18]:
coverage_summary = (
    processed_df
    .groupby(
        ["country_code", "country"],
        as_index=False,
    )
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        observations=("year", "size"),
    )
)

coverage_summary

,country_code,country,first_year,last_year,observations
0,GBR,United Kingdom,2010,2025,16
1,ITA,Italy,2010,2025,16
2,TUR,Turkiye,2010,2025,16


## Saving the processed dataset

The validated country-year table is saved under `data/processed/` so that
NB03 can load the analysis-ready file directly without repeating the raw
record extraction or reshaping.

In [19]:
processed_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print(
    f"Saved {OUTPUT_FILE.name} "
    f"({OUTPUT_FILE.stat().st_size:,} bytes)"
)

Saved country_year_indicators.csv (2,106 bytes)


### Verifying the saved output

The CSV is reloaded to inspect the file that was actually written to disk.
The subsequent checks confirm that it retains the expected shape and
columns and contains no missing values or duplicate country-year rows.

In [20]:
reloaded_df = pd.read_csv(OUTPUT_FILE)

print("Reloaded shape:", reloaded_df.shape)
print("\nReloaded types:")
print(reloaded_df.dtypes)

reloaded_preview = (
    reloaded_df
    .groupby(
        "country_code",
        sort=False,
    )
    .head(2)
    .reset_index(drop=True)
)

reloaded_preview

Reloaded shape: (48, 5)

Reloaded types:
country_code               object
country                    object
year                        int64
inflation_pct             float64
youth_unemployment_pct    float64
dtype: object


,country_code,country,year,inflation_pct,youth_unemployment_pct
0,GBR,United Kingdom,2010,2.492655,20.328
1,GBR,United Kingdom,2011,3.856112,21.689
2,ITA,Italy,2010,1.525516,27.910
3,ITA,Italy,2011,2.780633,29.145
4,TUR,Turkiye,2010,8.566444,21.578
5,TUR,Turkiye,2011,6.471880,18.208


In [21]:
if reloaded_df.shape != processed_df.shape:
    raise ValueError(
        "Reloaded CSV has an unexpected shape."
    )

if list(reloaded_df.columns) != list(processed_df.columns):
    raise ValueError(
        "Reloaded CSV has unexpected columns."
    )

if reloaded_df.isna().sum().sum() != 0:
    raise ValueError(
        "Reloaded CSV contains missing values."
    )

if reloaded_df.duplicated(
    subset=["country_code", "year"]
).sum() != 0:
    raise ValueError(
        "Reloaded CSV contains duplicate country-year rows."
    )

print("Verified:", OUTPUT_FILE.name)

Verified: country_year_indicators.csv


## Transformation summary

The two original World Bank API responses were combined into one
analysis-ready country-year table.

The final file contains 48 rows covering Italy, Türkiye, and the United
Kingdom from 2010 through 2025. Each row represents one country in one
year and contains:

- the annual consumer-price inflation rate;
- the modelled youth unemployment rate among the labour force aged 15–24;
- the country name and three-letter country code.

The table contains no missing values or duplicate country-year rows and
was saved as:

- `data/processed/country_year_indicators.csv`

NB03 will load this processed file directly and will not repeat the
raw-data extraction or reshaping performed here.